In [ ]:
# Instalar dependências (executar apenas se necessário)
%pip install sentence-transformers torch networkx scipy transformers --quiet

print("✅ Dependências instaladas!")


In [ ]:
# Imports necessários
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Adicionar scripts ao path
sys.path.append('scripts')

# Verificar disponibilidade de GPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️ Dispositivo: {device}")

# Configurar matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12


In [ ]:
# Executar experimento em modo teste
!python scripts/pheme_real_cascades_experiment.py --test


In [ ]:
# Demonstração de embeddings SBERT
from sentence_transformers import SentenceTransformer

# Carregar modelo SBERT
print("📥 Carregando SBERT...")
sbert_model = SentenceTransformer('all-mpnet-base-v2')

# Exemplos de textos
texts = [
    "Breaking: Multiple casualties reported in downtown shooting",
    "URGENT: Gunman opens fire in crowded area",  
    "Analysis: Understanding the history of urban violence",
    "Opinion: We must address gun control legislation",
    "Recipe: How to make the perfect chocolate cake"
]

# Gerar embeddings
embeddings = sbert_model.encode(texts, normalize_embeddings=True)

# Calcular similaridades
from sklearn.metrics.pairwise import cosine_similarity
similarities = cosine_similarity(embeddings)

# Visualizar matriz de similaridade
plt.figure(figsize=(8, 6))
sns.heatmap(similarities, annot=True, fmt='.2f', cmap='RdBu_r', center=0.5,
            xticklabels=[f"Texto {i+1}" for i in range(len(texts))],
            yticklabels=[f"Texto {i+1}" for i in range(len(texts))])
plt.title("Similaridade Semântica entre Textos (SBERT)")
plt.tight_layout()
plt.show()

# Mostrar textos
for i, text in enumerate(texts):
    print(f"Texto {i+1}: {text[:50]}...")


In [ ]:
# Demonstração de construção de TAG
from tag_construction import TAGConstructor
import networkx as nx

# Criar cascata simulada
cascade_texts = [
    "Breaking: Explosion reported at chemical plant",  # Post original
    "URGENT: Multiple explosions at industrial facility, evacuations underway",  # Mutação 1
    "Reports of toxic gas leak following plant explosion",  # Mutação 2
    "Authorities confirm controlled demolition, not explosion",  # Correção
    "Conspiracy theories emerge about plant 'explosion' coverup"  # Recombinação
]

# Timestamps simulados (em segundos)
timestamps = [0, 300, 600, 1200, 1800]  # 0, 5min, 10min, 20min, 30min

# Gerar embeddings da cascata
cascade_embeddings = sbert_model.encode(cascade_texts, normalize_embeddings=True)

# Construir TAG
tag_constructor = TAGConstructor(similarity_threshold=0.7)
tag = tag_constructor.build_tag(
    embeddings=cascade_embeddings,
    timestamps=timestamps,
    post_ids=[f"post_{i}" for i in range(len(cascade_texts))]
)

# Visualizar o grafo
plt.figure(figsize=(10, 8))
pos = nx.spring_layout(tag, k=2, iterations=50)

# Desenhar nós e arestas
nx.draw_networkx_nodes(tag, pos, node_size=1000, node_color='lightblue', edgecolors='black')
nx.draw_networkx_edges(tag, pos, edge_color='gray', arrows=True, arrowsize=20, width=2)

# Labels
labels = {f"post_{i}": f"P{i}" for i in range(len(cascade_texts))}
nx.draw_networkx_labels(tag, pos, labels, font_size=14)

# Adicionar pesos das arestas (similaridade)
edge_labels = {}
for u, v, data in tag.edges(data=True):
    if 'weight' in data:
        edge_labels[(u, v)] = f"{data['weight']:.2f}"
nx.draw_networkx_edge_labels(tag, pos, edge_labels, font_size=10)

plt.title("Tree Alignment Graph (TAG) da Cascata")
plt.axis('off')
plt.tight_layout()
plt.show()

# Mostrar textos
print("📝 Textos da cascata:")
for i, text in enumerate(cascade_texts):
    print(f"P{i}: {text}")


In [ ]:
# Extrair características filogenéticas
features_df = tag_constructor.extract_phylogenetic_features(tag)

# Remover colunas não-numéricas para visualização
numeric_features = features_df.drop(['node_id', 'community_id'], axis=1)

# Visualizar características como heatmap
plt.figure(figsize=(12, 6))
sns.heatmap(numeric_features.T, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=[f"P{i}" for i in range(len(features_df))],
            yticklabels=numeric_features.columns,
            cbar_kws={'label': 'Valor'})
plt.title("Características Filogenéticas Extraídas do TAG")
plt.xlabel("Posts")
plt.ylabel("Características")
plt.tight_layout()
plt.show()

# Estatísticas das características
print("📊 Estatísticas das características filogenéticas:")
print(numeric_features.describe().round(2))
